In [2]:
import google.auth
import numpy as np
import pandas as pd
import geopandas as gpd
from calitp_data_analysis import geography_utils
from calitp_data_analysis.sql import to_snakecase
from shared_utils import arcgis_query

In [3]:

import re
import pyarrow.parquet as pq
import gcsfs

In [4]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [5]:
from calitp_data_analysis import get_fs
fs = get_fs()

In [6]:

import os
from typing import List, Optional, Union
import pyarrow.dataset as ds
from google.cloud import storage

In [7]:
import google.auth
import pandas_gbq

credentials, project = google.auth.default()
from functools import cache

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas
from calitp_data_analysis.sql import to_snakecase

In [8]:
from typing import List

In [9]:
import geopandas as gpd
import pandas as pd

In [10]:
from functools import cache

from calitp_data_analysis.gcs_geopandas import GCSGeoPandas

@cache
def gcs_geopandas():
    return GCSGeoPandas()

In [11]:
@cache
def gcs_pandas():
    return GCSPandas()

In [12]:
gcsgp = GCSGeoPandas()

# Census Blocks

In [13]:
def load_hpms(url:str)->gpd.GeoDataFrame:
    df = to_snakecase(gcsgp.read_parquet(
    hpms_url)).to_crs(geography_utils.CA_NAD83Albers_m )
    return df

In [14]:
census_block_url = "gs://calitp-analytics-data/data-analyses/equity_index/unbuffered_censusblocks_2020/combined_unbufferd_ca_2020.parquet"

In [15]:
census_df = to_snakecase(gcsgp.read_parquet(
    census_block_url)).to_crs(geography_utils.CA_NAD83Albers_m )

In [16]:
# 

## HPMS

In [17]:


hpms_url = "gs://calitp-analytics-data/data-analyses/equity_index/HPMS21_Main_SUCU.parquet"



In [18]:
hpm_df = load_hpms(hpms_url)

In [19]:
hpm_df.f_system.unique()

array([7, 5, 4, 3, 6, 2, 1], dtype=int32)

In [20]:
# Filter to f_system 1,2 temporarily
hpm_df2 = hpm_df.loc[hpm_df.f_system.isin([1,2])]

## Main function

In [21]:
buffer_list = [500, 450, 400, 350, 300, 250, 200, 150, 100, 50]

In [22]:
def buffer_intersect(hpm_gdf:gpd.GeoDataFrame, census_gdf:gpd.GeoDataFrame, buffer: int)->pd.DataFrame:
    hpm_gdf.geometry = hpm_gdf.geometry.buffer(buffer)
    
    intersect = gpd.overlay(hpm_gdf, census_gdf, how = "intersection",keep_geom_type= False)

    # Find area 
    intersect["area"] = intersect.geometry.area
    
    # Create an unique ID 
    intersect["unique_id"] = intersect.geoid20 + "_" + intersect.routeid

    # Find max value per unique ID
    intersect[f"aadt_max"] = intersect.groupby("unique_id")["aadt"].transform("max")

    # Sum the maximum AADT by GEOID
    agg = intersect.groupby("geoid20").agg({f"aadt_max":"sum", "area":"max"}).reset_index()

    # Calculate weighted aadt
    score_col_name = f"aadt_{buffer}_score"
    agg[score_col_name] = agg.aadt_max * agg.area
    agg2 = agg.groupby("geoid20").agg({score_col_name:"sum"}).reset_index()

    # Save to GCS
    agg2.to_parquet("./agg2.parquet")
    fs.put("./agg2.parquet", f"calitp-analytics-data/data-analyses/equity_index/eqi_traffic_{buffer}.parquet")
    print(f"finshed saving data for {buffer}")
    return agg2

In [ ]:
# buffer_500 = buffer_intersect(hpm_gdf=hpm_df2, census_gdf=census_gdf, buffer = buffer_list[0])

In [24]:
for buffer in [200,150,100,50]:
   df = buffer_intersect(hpm_gdf=hpm_df2, census_gdf=census_df, buffer = buffer)

/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


finshed saving data for 200


/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


finshed saving data for 150


/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


finshed saving data for 100


/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


finshed saving data for 50


In [ ]:
df = gcs_pandas().read_parquet("gs://calitp-analytics-data/data-analyses/equity_index/eqi_traffic_100.parquet")

In [ ]:
df.sample()

In [25]:

def load_eqi_traffic_parquets():
    """
    Reads all EQI traffic parquet files from the GCS folder
    and concatenates them together on the column 'geoid20'.
    """

    base_path = "gs://calitp-analytics-data/data-analyses/equity_index/eqi_traffic_"
    sizes = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500]

    dfs = []

    for size in sizes:
        path = f"{base_path}{size}.parquet"
        df = gcs_pandas().read_parquet(path)
        dfs.append(df)

   # Merge all dfs on geoid20
    combined = dfs[0]
    for df in dfs[1:]:
        combined = combined.merge(df, on="geoid20", how="outer")


    # Optional: ensure geoid20 exists
    if "geoid20" not in combined.columns:
        raise ValueError("Column 'geoid20' not found in concatenated data.")

    return combined


In [26]:
df = load_eqi_traffic_parquets()

In [ ]:
df.head(3)

In [ ]:
df.columns

In [27]:
df["p_50_m"]  = df["aadt_50_score"]

In [28]:
df["p_100_m"] = df["aadt_100_score"] - df["aadt_50_score"]

In [29]:
df["p_150_m"] = df["aadt_150_score"] - df["aadt_100_score"]

In [30]:
df["p_200_m"] = df["aadt_200_score"] - df["aadt_150_score"]

In [31]:
df["p_250_m"] = df["aadt_250_score"] - df["aadt_200_score"]

In [32]:
df["p_300_m"] = df["aadt_300_score"] - df["aadt_250_score"]

In [33]:
df["p_350_m"] = df["aadt_350_score"] - df["aadt_250_score"]

In [34]:
df["p_400_m"] = df["aadt_400_score"] - df["aadt_350_score"]

In [35]:
df["p_450_m"] = df["aadt_450_score"] - df["aadt_400_score"]

In [36]:
df["p_500_m"] = df["aadt_500_score"] - df["aadt_450_score"]

In [37]:
weights = {
	        "p_50_m": 1.0,
	        "p_100_m": 0.5,
	        "p_150_m": 0.33,
	        "p_200_m": 0.25,
	        "p_250_m": 0.20,
	        "p_300_m": 0.16,
	        "p_350_m": 0.14,
	        "p_400_m": 0.125,
	        "p_450_m": 0.111,
	        "p_500_m": 0.10,
    }

In [38]:
for col, w in weights.items():
    df[f"{col}_w"] = df[col] * w

In [39]:
weighted_cols = [f"{col}_w" for col in weights.keys()]
df["weighted_aadt_score"] = df[weighted_cols].sum(axis=1)

In [40]:
df.head(2)

,geoid20,aadt_50_score,aadt_100_score,aadt_150_score,aadt_200_score,aadt_250_score,aadt_300_score,aadt_350_score,aadt_400_score,aadt_450_score,aadt_500_score,p_50_m,p_100_m,p_150_m,p_200_m,p_250_m,p_300_m,p_350_m,p_400_m,p_450_m,p_500_m,p_50_m_w,p_100_m_w,p_150_m_w,p_200_m_w,p_250_m_w,p_300_m_w,p_350_m_w,p_400_m_w,p_450_m_w,p_500_m_w,weighted_aadt_score
0,060014001001010,161853430380.49,132415367151.33,47120639761.52,9590153862.23,512775767.18,NaN,NaN,320689999401.06,12439681599.75,NaN,161853430380.49,-29438063229.15,-85294727389.81,-37530485899.30,-9077378095.04,NaN,NaN,NaN,-308250317801.31,NaN,161853430380.49,-14719031614.58,-28147260038.64,-9382621474.82,-1815475619.01,NaN,NaN,NaN,-34215785275.95,NaN,73573256357.49
1,060014001001011,901239215714.64,661378044203.45,311152293143.04,90518663695.67,5199933673.75,NaN,NaN,1537040151476.03,149402168626.19,NaN,901239215714.64,-239861171511.19,-350225751060.42,-220633629447.36,-85318730021.92,NaN,NaN,NaN,-1387637982849.84,NaN,901239215714.64,-119930585755.59,-115574497849.94,-55158407361.84,-17063746004.38,NaN,NaN,NaN,-154027816096.33,NaN,439484162646.55


In [41]:
df2 = df[["geoid20", "weighted_aadt_score"]].copy().fillna(0)

In [42]:
df2["traffic_proximity_and_volume_percentile"] = (
	        df2["weighted_aadt_score"].rank(pct=True)
    )

In [43]:
df2.head()

,geoid20,weighted_aadt_score,traffic_proximity_and_volume_percentile
0,060014001001010,73573256357.49,0.88
1,060014001001011,439484162646.55,0.98
2,060014001001012,57223265937.04,0.85
3,060014001001013,1159671584.76,0.16
4,060014001001014,0.00,0.03


In [44]:
df2.weighted_aadt_score.describe()

count           213932.00
mean       49528657073.92
std       241227627629.00
min       -17638489593.61
25%         2747382364.07
50%        10471092117.32
75%        31938224538.79
max     30872815895421.31
Name: weighted_aadt_score, dtype: float64

In [45]:
df2.traffic_proximity_and_volume_percentile.describe()

count   213932.00
mean         0.50
std          0.29
min          0.00
25%          0.25
50%          0.50
75%          0.75
max          1.00
Name: traffic_proximity_and_volume_percentile, dtype: float64

In [48]:
m1 = pd.merge(census_df[["geoid20","county_name", "geometry"]], df2, on = ["geoid20"], how = "left")

In [49]:
m1.shape

(519723, 5)

In [50]:
m1.columns

Index(['geoid20', 'county_name', 'geometry', 'weighted_aadt_score',
       'traffic_proximity_and_volume_percentile'],
      dtype='object')

In [51]:
m1.county_name.unique()

array(['Alameda', 'Alpine', 'Amador', 'Butte', 'Calaveras', 'Colusa',
       'Contra Costa', 'Del Norte', 'El Dorado', 'Fresno', 'Glenn',
       'Humboldt', 'Imperial', 'Inyo', 'Kern', 'Kings', 'Lake', 'Lassen',
       'Los Angeles', 'Madera', 'Marin', 'Mariposa', 'Mendocino',
       'Merced', 'Modoc', 'Mono', 'Monterey', 'Napa', 'Nevada', 'Orange',
       'Placer', 'Plumas', 'Riverside', 'Sacramento', 'San Benito',
       'San Bernardino', 'San Diego', 'San Francisco', 'San Joaquin',
       'San Luis Obispo', 'San Mateo', 'Santa Barbara', 'Santa Clara',
       'Santa Cruz', 'Shasta', 'Sierra', 'Siskiyou', 'Solano', 'Sonoma',
       'Stanislaus', 'Sutter', 'Tehama', 'Trinity', 'Tulare', 'Tuolumne',
       'Ventura', 'Yolo', 'Yuba'], dtype=object)

In [53]:
m1.loc[(m1.county_name == "San Francisco")&(m1.traffic_proximity_and_volume_percentile > 0.9)].shape

(138, 5)

In [61]:
m1.loc[(m1.county_name == "San Francisco")][["traffic_proximity_and_volume_percentile"]].describe()

,traffic_proximity_and_volume_percentile
count,3452.00
mean,0.46
std,0.27
min,0.00
25%,0.19
50%,0.49
75%,0.68
max,1.00


In [72]:
# m1.loc[(m1.county_name == "San Francisco")].explore("traffic_proximity_and_volume_percentile")